In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde
import diptest

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"\nInput file not found:\n{INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

required_columns = ["patient_id", "median_thickness_nm"]
missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n" + "\n".join(missing_columns)
    )

patients = df["patient_id"].dropna().unique()

print()
print("GBM MULTIMODALITY ANALYSIS")
print("=" * 70)
print(f"Total membrane components : {len(df)}")
print(f"Patients found            : {len(patients)}")
print("=" * 70)

summary_results = []

for PATIENT_ID in patients:

    print()
    print("-" * 70)
    print(f"Processing Patient {PATIENT_ID}")
    print("-" * 70)

    thickness = (
        df[df["patient_id"] == PATIENT_ID]["median_thickness_nm"]
        .dropna()
        .to_numpy()
    )

    n_samples = len(thickness)
    print(f"Membrane components : {n_samples}")

    if n_samples < 5:
        print("Skipped - too few membrane components")
        continue

    X = thickness.reshape(-1, 1)

    # Dip Test
    dip_statistic, dip_pvalue = diptest.diptest(thickness)
    dip_classification = "Multimodal" if dip_pvalue < 0.05 else "Unimodal"

    # Fit GMM
    models = {}
    bic_values = {}
    aic_values = {}

    for n_components in [1, 2, 3]:
        model = GaussianMixture(
            n_components=n_components, random_state=42, n_init=20
        )
        model.fit(X)
        models[n_components] = model
        bic_values[n_components] = model.bic(X)
        aic_values[n_components] = model.aic(X)

    best_components = min(bic_values, key=bic_values.get)
    gmm = models[best_components]

    original_means = gmm.means_.flatten()
    original_weights = gmm.weights_.flatten()

    component_order = np.argsort(original_means)
    component_means = original_means[component_order]
    component_weights = original_weights[component_order]

    # Interpretation logic
    if dip_classification == "Unimodal":
        if best_components == 1:
            gmm_interpretation = "Unimodal; single GMM component"
        else:
            gmm_interpretation = "Unimodal; GMM subcomponents"
    else:
        if best_components == 1:
            gmm_interpretation = "Multimodal by Dip Test;\nsingle GMM component"
        else:
            gmm_interpretation = "Multimodal"

    print(f"Dip statistic      : {dip_statistic:.5f}")
    print(f"Dip p-value        : {dip_pvalue:.5f}")
    print(f"Dip classification : {dip_classification}")
    print(f"Best GMM           : {best_components} component(s)")
    print(f"BIC                : {bic_values[best_components]:.2f}")
    print(f"AIC                : {aic_values[best_components]:.2f}")

    for i in range(best_components):
        print(
            f"Component {i + 1}       : Mean = {component_means[i]:.2f} nm, "
            f"Weight = {component_weights[i]:.3f}"
        )

    print(f"Interpretation     : {gmm_interpretation}")

    result = {
        "Patient_ID": PATIENT_ID,
        "Number_of_membranes": n_samples,
        "Dip_statistic": dip_statistic,
        "Dip_p_value": dip_pvalue,
        "Dip_classification": dip_classification,
        "Best_GMM_components": best_components,
        "BIC_1": bic_values[1],
        "BIC_2": bic_values[2],
        "BIC_3": bic_values[3],
        "AIC_1": aic_values[1],
        "AIC_2": aic_values[2],
        "AIC_3": aic_values[3],
        "Best_BIC": bic_values[best_components],
        "Best_AIC": aic_values[best_components],
        "GMM_interpretation": gmm_interpretation,
    }

    for i in range(3):
        if i < best_components:
            result[f"Peak_{i + 1}_nm"] = component_means[i]
            result[f"Weight_{i + 1}"] = component_weights[i]
        else:
            result[f"Peak_{i + 1}_nm"] = np.nan
            result[f"Weight_{i + 1}"] = np.nan

    summary_results.append(result)

    x = np.linspace(thickness.min() - 50, thickness.max() + 50, 1000)
    x_plot = x.reshape(-1, 1)
    total_density = np.exp(gmm.score_samples(x_plot))

    fig, ax = plt.subplots(figsize=(13, 7))

    # Histogram
    ax.hist(
        thickness,
        bins=15,
        density=True,
        alpha=0.30,
        color="#1f77b4",
        edgecolor="#1f77b4",
        label="Observed thickness",
    )

    # KDE
    try:
        kde = gaussian_kde(thickness)
        ax.plot(x, kde(x), linewidth=2, color="#ff7f0e", label="KDE")
    except np.linalg.LinAlgError:
        print("KDE could not be calculated.")

    # Total GMM Density
    ax.plot(
        x,
        total_density,
        linewidth=2.5,
        color="#2ca02c",
        label=f"Total GMM ({best_components} components)",
    )

    colors = ["#d62728", "#9467bd", "#8c564b"]
    for i in range(best_components):
        mean = component_means[i]
        weight = component_weights[i]
        original_index = component_order[i]
        std = np.sqrt(gmm.covariances_[original_index][0][0])

        gaussian = (
            weight
            * (1 / (std * np.sqrt(2 * np.pi)))
            * np.exp(-0.5 * ((x - mean) / std) ** 2)
        )

        c = colors[i % len(colors)]

        ax.plot(
            x,
            gaussian,
            linestyle="--",
            linewidth=1.8,
            color=c,
            label=f"GMM comp {i + 1}: {mean:.1f} nm (w={weight:.2f})",
        )

        ax.axvline(
            mean,
            linestyle=":",
            linewidth=1.5,
            color=c,
            alpha=0.85,
            label=f"Mean μ{i + 1} = {mean:.1f} nm",
        )
  
    information_text = (
        "      Image Information\n"
        f"Patient ID: {PATIENT_ID}\n"
        f"Membrane components: {n_samples}\n\n"
        f"Dip statistic: {dip_statistic:.4f}\n"
        f"Dip p-value: {dip_pvalue:.4f}\n"
        f"Classification: {dip_classification}\n\n"
        f"Best GMM: {best_components} components\n"
        f"BIC: {bic_values[best_components]:.2f}\n"
        f"AIC: {aic_values[best_components]:.2f}\n\n"
        f"Interpretation:\n{gmm_interpretation}"
    )
    
    ax.text(
        1.03,
        1.0,
        information_text,
        transform=ax.transAxes,
        verticalalignment="top",
        horizontalalignment="left",
        fontsize=9.5,
        linespacing=1.3,
        bbox=dict(
            boxstyle="round,pad=0.6",
            facecolor="none",
            edgecolor="#444444",
            linewidth=0.8,
            alpha=0.9,
        ),
    )

    legend = ax.legend(
        loc="lower left",
        bbox_to_anchor=(1.03, 0.0),  
        borderaxespad=0,
        fontsize=9,
        frameon=True,
        facecolor="none",
        edgecolor="#444444",
        title="Distribution Details",
        title_fontsize=10,
    )

    ax.set_xlabel("GBM thickness (nm)", fontsize=12, labelpad=8)
    ax.set_ylabel("Density", fontsize=12, labelpad=8)
    ax.set_title(
        f"GMM Multimodality Analysis — Patient {PATIENT_ID}",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )

    ax.grid(True, alpha=0.15, linestyle="--", linewidth=0.7)
    
    plt.subplots_adjust(left=0.08, right=0.72, top=0.90, bottom=0.12)

    output_file = os.path.join(OUTPUT_FOLDER, f"GMM_patient_{PATIENT_ID}.png")

    plt.savefig(output_file, dpi=300, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)

    print("Plot saved:", output_file)

summary_df = pd.DataFrame(summary_results)

if summary_df.empty:
    print("\nNo valid patients were analysed.")
    raise SystemExit

summary_df = summary_df.sort_values(
    by=["Dip_p_value", "Best_GMM_components", "Best_BIC"],
    ascending=[True, False, True],
).reset_index(drop=True)

summary_df.insert(0, "Rank", np.arange(1, len(summary_df) + 1))

summary_file = os.path.join(OUTPUT_FOLDER, "GMM_all_patients_summary.csv")

try:
    summary_df.to_csv(summary_file, index=False)
    print("\nSummary CSV saved:\n", summary_file)
except PermissionError:
    print("\nWARNING: Summary CSV is locked by another process.")

print("\nALL PATIENT GMM ANALYSIS COMPLETED")
print("=" * 70)


GBM MULTIMODALITY ANALYSIS
Total membrane components : 394
Patients found            : 11

----------------------------------------------------------------------
Processing Patient 01-24
----------------------------------------------------------------------
Membrane components : 19
Dip statistic      : 0.06017
Dip p-value        : 0.85855
Dip classification : Unimodal
Best GMM           : 3 component(s)
BIC                : 204.32
AIC                : 196.76
Component 1       : Mean = 170.83 nm, Weight = 0.842
Component 2       : Mean = 379.97 nm, Weight = 0.053
Component 3       : Mean = 480.62 nm, Weight = 0.105
Interpretation     : Unimodal; GMM subcomponents
Plot saved: C:\Users\ishin\OneDrive\Desktop\ish\GMM_plots\GMM_patient_01-24.png

----------------------------------------------------------------------
Processing Patient 02-24
----------------------------------------------------------------------
Membrane components : 41
Dip statistic      : 0.03271
Dip p-value        : 0.991